# trca_visualization

This notebook loads saved EEG trial files for the three conditions, reconstructs the original trial order for each run, applies the same baseline cropping used in the training scripts, runs FBTRCA-based classification, and visualizes accuracy and ITR across conditions.

Outputs:
- run-level summary table
- condition-level summary table
- accuracy plots
- ITR plots

In [1]:
import sys
print("Python:", sys.executable)

try:
    import brainda
    print("brainda is installed")
except Exception as e:
    print("brainda import failed:", e)

Python: /opt/conda/bin/python
brainda is installed


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
from sklearn.pipeline import clone
from collections import OrderedDict

from brainda.algorithms.utils.model_selection import (
    set_random_seeds,
    generate_loo_indices,
    match_loo_indices
)
from brainda.algorithms.decomposition import (
    FBTRCA,
    generate_filterbank
)

In [3]:
print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
print(os.listdir("."))

Current working directory:
/home/n1shibuya/OpenVEP/trca_based_analysis

Files and folders here:
['trca_data_viz.ipynb', 'trca_stats.ipynb', '.ipynb_checkpoints']


### Basic settings

In [4]:
# Folder paths
PROJECT_ROOT = os.path.abspath("..")

normal_folder = os.path.join(
    PROJECT_ROOT, "data", "cyton8_alternating-vep_32-class_1.2s", "sub-01", "ses-02"
)
redgreen_folder = os.path.join(
    PROJECT_ROOT, "data", "redgreen_cyton8_alternating-vep_32-class_1.2s", "sub-01", "ses-01"
)
blueyellow_folder = os.path.join(
    PROJECT_ROOT, "data", "blueyellow_cyton8_alternating-vep_32-class_1.2s", "sub-01", "ses-01"
)

# Basic experiment settings
sampling_rate = 250
stim_duration = 1.2
baseline_duration = 0.2
baseline_samples = int(baseline_duration * sampling_rate)

n_classes = 32
n_per_class = 2
trial_seconds_for_itr = stim_duration + 0.7  # edit if your actual trial time differs

# 32 stimulus classes used in your current scripts
stimulus_classes = [
    (8, 0), (8, 0.5), (8, 1), (8, 1.5),
    (9, 0), (9, 0.5), (9, 1), (9, 1.5),
    (10, 0), (10, 0.5), (10, 1), (10, 1.5),
    (11, 0), (11, 0.5), (11, 1), (11, 1.5),
    (12, 0), (12, 0.5), (12, 1), (12, 1.5),
    (13, 0), (13, 0.5), (13, 1), (13, 1.5),
    (14, 0), (14, 0.5), (14, 1), (14, 1.5),
    (15, 0), (15, 0.5), (15, 1), (15, 1.5),
]

target_tab = {tuple(map(float, cls)): idx for idx, cls in enumerate(stimulus_classes)}

### Helper functions

In [5]:
def list_run_files(folder_path):
    """
    Return run files sorted by run number.

    Expected file name format:
    eeg-trials_2-per-class_run-{run}.npy
    """
    run_files = [
        f for f in os.listdir(folder_path)
        if f.startswith("eeg-trials_2-per-class_run-") and f.endswith(".npy")
    ]

    def extract_run_number(filename):
        return int(filename.split("-")[-1].split(".")[0])

    run_files = sorted(run_files, key=extract_run_number)
    return run_files


def load_and_reconstruct_one_run(folder_path, run_file):
    """
    Load one saved EEG trial file and reconstruct the original trial order.

    Why this is needed:
    In the experiment script, the trial sequence is shuffled using the run number
    as the random seed. In the training scripts, that same seed is used to recover
    the original order. We repeat that here so the notebook matches train_trca*.py.

    Output shape after reshape:
    (2, 32, 8, 350)
    meaning:
    - 2 repetitions per class
    - 32 stimulus classes
    - 8 EEG channels
    - 350 samples per trial (0.2 s baseline + 1.2 s stimulation at 250 Hz)
    """
    run_number = int(run_file.split("-")[-1].split(".")[0])
    eeg_trials = np.load(os.path.join(folder_path, run_file), allow_pickle=True)

    eeg_trials = np.array(eeg_trials)

    # Recover original ordering
    np.random.seed(run_number)
    shuffled_indices = np.random.permutation(eeg_trials.shape[0])

    reverted_eeg_trials = np.empty_like(eeg_trials)
    reverted_eeg_trials[shuffled_indices] = eeg_trials

    # Match the training script shape
    reverted_eeg_trials = reverted_eeg_trials.reshape(2, 32, 8, 350)

    return run_number, reverted_eeg_trials


def crop_baseline(eeg_4d, baseline_samples=50):
    """
    Remove the baseline portion from each trial.

    Input shape:
    (n_repetitions, 32, 8, 350)

    Output shape:
    (n_repetitions, 32, 8, 300)
    """
    return eeg_4d[:, :, :, baseline_samples:]


def itr_bits_per_minute(accuracy, n_classes=32, trial_seconds=1.9):
    """
    Compute Information Transfer Rate (ITR) in bits/min.

    Formula:
    ITR = [log2(N) + P*log2(P) + (1-P)*log2((1-P)/(N-1))] * 60 / T

    where:
    - N = number of classes
    - P = accuracy
    - T = time per selection in seconds

    Edge cases:
    - If P <= 0, return 0
    - If P >= 1, use the simplified perfect-accuracy form
    """
    P = float(accuracy)
    N = int(n_classes)
    T = float(trial_seconds)

    if P <= 0:
        return 0.0
    if P >= 1:
        return np.log2(N) * 60.0 / T

    term1 = np.log2(N)
    term2 = P * np.log2(P)
    term3 = (1 - P) * np.log2((1 - P) / (N - 1))
    return (term1 + term2 + term3) * 60.0 / T

### FBTRCA function

This is a simplified notebook version of the logic already used in `train_trca.py`.
It computes run-level accuracy and confusion matrices.

In [6]:
def run_fbtrca_simple(
    eeg,
    target_by_trial,
    target_tab,
    duration=1.2,
    onset_delay=0,
    srate=250,
    ensemble=True,
    print_acc=False
):
    """
    Run FBTRCA classification using leave-one-out evaluation.

    Input eeg shape:
    (n_trials_per_class, 32, 8, n_samples)

    Important:
    This follows the same broad structure as the uploaded training scripts:
    - shuffle eeg with a fixed seed
    - flatten into class/trial samples
    - build FBTRCA filterbank
    - leave-one-out evaluation
    - return confusion matrix and accuracy

    Returns:
    - normalized confusion matrix
    - overall accuracy
    """
    eeg = np.copy(eeg)

    # Match the fixed seed behavior in the training script
    np.random.seed(64)
    np.random.shuffle(eeg)

    n_trials = eeg.shape[0]
    classes = range(32)
    n_classes_local = len(classes)

    # Labels: for each repetition, classes 0..31
    y = np.array([list(target_tab.values())] * n_trials).T.reshape(-1)

    # Keep all classes, all channels, crop onset if needed
    eeg_temp = eeg[:n_trials, classes, :, onset_delay:]

    # Reshape to sample-wise form used by the classifier
    # from (repetition, class, channel, time)
    # to   (repetition*class, channel, time)
    X = eeg_temp.swapaxes(0, 1).reshape(-1, *eeg_temp.shape[2:])

    # Filterbank settings copied from the train_trca scripts
    n_bands = 3
    wp = [[8 * i, 90] for i in range(1, n_bands + 1)]
    ws = [[8 * i - 2, 95] for i in range(1, n_bands + 1)]
    filterbank = generate_filterbank(wp, ws, srate, order=4, rp=1)
    filterweights = np.arange(1, len(filterbank) + 1) ** (-1.25) + 0.25

    set_random_seeds(64)

    models = OrderedDict([
        ("fbtrca", FBTRCA(filterbank, filterweights=filterweights, ensemble=ensemble))
    ])

    # Event labels for leave-one-out indexing
    events = []
    for j_class in classes:
        events.extend([str(target_by_trial[i_trial][j_class]) for i_trial in range(n_trials)])
    events = np.array(events)

    subjects = ["1"] * (n_classes_local * n_trials)
    meta = pd.DataFrame(
        data=np.array([subjects, events]).T,
        columns=["subject", "event"]
    )

    set_random_seeds(42)
    loo_indices = generate_loo_indices(meta)

    # We only have one model here
    model_name, base_model = list(models.items())[0]

    filterX = np.copy(X[..., :int(srate * duration)])
    filterY = np.copy(y)

    # Mean-center each trial
    filterX = filterX - np.mean(filterX, axis=-1, keepdims=True)

    n_loo = len(loo_indices["1"][events[0]])

    loo_accs = []
    testYs = []
    pred_labelss = []

    for k in range(n_loo):
        train_ind, validate_ind, test_ind = match_loo_indices(k, meta, loo_indices)
        train_ind = np.concatenate([train_ind, validate_ind])

        trainX, trainY = filterX[train_ind], filterY[train_ind]
        testX, testY = filterX[test_ind], filterY[test_ind]

        model = clone(base_model)
        model.fit(trainX, trainY)

        pred_labels = model.predict(testX)

        loo_accs.append(balanced_accuracy_score(testY, pred_labels))
        pred_labelss.extend(pred_labels)
        testYs.extend(testY)

    acc = accuracy_score(testYs, pred_labelss)
    cm = confusion_matrix(testYs, pred_labelss, normalize="true")

    if print_acc:
        print(f"Model: {model_name}, LOO Acc: {np.mean(loo_accs):.4f}, Overall Acc: {acc:.4f}")

    return cm, acc

### Run-level summary

In [7]:
def evaluate_one_condition(folder_path, condition_name):
    """
    Evaluate all runs in one condition folder.

    For each run:
    1. load EEG trial data
    2. restore original trial order
    3. remove baseline samples
    4. run FBTRCA classification
    5. compute ITR

    Returns:
    - run_summary_df
    - confusion_matrices_by_run
    """
    run_files = list_run_files(folder_path)

    run_rows = []
    confusion_matrices = {}

    for run_file in run_files:
        run_number, eeg_4d = load_and_reconstruct_one_run(folder_path, run_file)

        cropped_eeg = crop_baseline(eeg_4d, baseline_samples=baseline_samples)

        # For 2 repetitions per class, target_by_trial must have length 2
        target_by_trial = [stimulus_classes] * cropped_eeg.shape[0]

        cm, acc = run_fbtrca_simple(
            eeg=cropped_eeg,
            target_by_trial=target_by_trial,
            target_tab=target_tab,
            duration=stim_duration,
            onset_delay=0,
            srate=sampling_rate,
            ensemble=True,
            print_acc=True
        )

        itr = itr_bits_per_minute(
            accuracy=acc,
            n_classes=n_classes,
            trial_seconds=trial_seconds_for_itr
        )

        row = {
            "condition": condition_name,
            "run": run_number,
            "accuracy": acc,
            "ITR": itr,
            "n_trials_per_class": cropped_eeg.shape[0],
            "n_total_selections": cropped_eeg.shape[0] * n_classes
        }
        run_rows.append(row)
        confusion_matrices[run_number] = cm

    run_summary_df = pd.DataFrame(run_rows).sort_values(["condition", "run"]).reset_index(drop=True)
    return run_summary_df, confusion_matrices

In [8]:
normal_summary, normal_cms = evaluate_one_condition(normal_folder, "Normal")
redgreen_summary, redgreen_cms = evaluate_one_condition(redgreen_folder, "Red-Green")
blueyellow_summary, blueyellow_cms = evaluate_one_condition(blueyellow_folder, "Blue-Yellow")

run_summary = pd.concat(
    [normal_summary, redgreen_summary, blueyellow_summary],
    axis=0,
    ignore_index=True
)

run_summary

ValueError: The groups parameter contains fewer than 2 unique groups ([1]). LeaveOneGroupOut expects at least 2.

In [ ]:
os.makedirs("results", exist_ok=True)

run_summary.to_csv("results/run_summary_simple.csv", index=False)

condition_summary = (
    run_summary
    .groupby("condition", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_ITR=("ITR", "mean"),
        std_ITR=("ITR", "std"),
        n_runs=("run", "count")
    )
)

condition_summary.to_csv("results/condition_summary_simple.csv", index=False)

print("Saved:")
print("- results/run_summary_simple.csv")
print("- results/condition_summary_simple.csv")

condition_summary

### Accuracy plot

In [ ]:
condition_order = ["Normal", "Red-Green", "Blue-Yellow"]

mean_acc = [
    run_summary.loc[run_summary["condition"] == cond, "accuracy"].mean()
    for cond in condition_order
]

plt.figure(figsize=(8, 5))
plt.bar(condition_order, mean_acc)
plt.ylabel("Accuracy")
plt.title("Mean Accuracy by Condition")
plt.ylim(0, max(mean_acc) * 1.25 if max(mean_acc) > 0 else 1)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for cond in condition_order:
    subset = run_summary[run_summary["condition"] == cond]
    plt.scatter([cond] * len(subset), subset["accuracy"], s=60)

plt.ylabel("Accuracy")
plt.title("Run-level Accuracy by Condition")
plt.show()

### IRT

In [ ]:
mean_itr = [
    run_summary.loc[run_summary["condition"] == cond, "ITR"].mean()
    for cond in condition_order
]

plt.figure(figsize=(8, 5))
plt.bar(condition_order, mean_itr)
plt.ylabel("ITR (bits/min)")
plt.title("Mean ITR by Condition")
plt.ylim(0, max(mean_itr) * 1.25 if max(mean_itr) > 0 else 1)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for cond in condition_order:
    subset = run_summary[run_summary["condition"] == cond]
    plt.scatter([cond] * len(subset), subset["ITR"], s=60)

plt.ylabel("ITR (bits/min)")
plt.title("Run-level ITR by Condition")
plt.show()

### Condition summary table

In [ ]:
condition_summary